In [1]:
import jax
import jax.numpy as jnp
from custom_jax import forces
import numpy as np
import time

Running cmake --build & --install in /home/jens/repos/custom-jax/build


# Check validity of implementations

In [2]:
x = jax.random.normal(jax.random.PRNGKey(0), (1024, 3)).block_until_ready()

phi0 = forces.potential_pure_jax_jit(x, eps=1e-2)
phi1 = forces.potential_jit(x, block_size=32, eps=1e-2)
phi2 = forces.potential_jit(x, block_size=64, eps=1e-2)

print(np.allclose(phi0, phi1, rtol=1e-4))
print(np.allclose(phi1, phi2, rtol=1e-4))

acc0 = forces.force_pure_jax_jit(x, eps=1e-2)
acc1 = forces.force_jit(x, block_size=32, eps=1e-2)
acc2 = forces.force_jit(x, block_size=64, eps=1e-2)

print(np.allclose(acc0, acc1, rtol=1e-3))
print(np.allclose(acc1, acc2, rtol=1e-3))

True
True
True
True


# Profile Potential

In [6]:
%timeit -n 10 x = 2

35.8 ns ± 42.9 ns per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
x = jax.random.normal(jax.random.PRNGKey(0), (128*1024, 3)).block_until_ready()
%timeit -n 10 forces.potential_pure_jax_jit(x).block_until_ready()
%timeit -n 10 forces.potential_jit(x, block_size=16).block_until_ready() # Never use blocksize < 32. Not all threads in a warp will be used.
%timeit -n 10 forces.potential_jit(x, block_size=32).block_until_ready()
%timeit -n 10 forces.potential_jit(x, block_size=64).block_until_ready()
%timeit -n 10 forces.potential_jit(x, block_size=128).block_until_ready()
%timeit -n 10 forces.potential_jit(x, block_size=256).block_until_ready()

t0 = time.time()
for i in range(10):
    forces.potential_jit(x, block_size=64).block_until_ready()
t1 = time.time()

dt = (t1-t0) / 10
print(f"{dt*1e3:.1f}ms =>  {(128*1024)**2 / dt  / 1e9:.1f} Bio. interactions per second", )

144 ms ± 6.24 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
72.3 ms ± 1.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
41.5 ms ± 556 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
41 ms ± 512 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
40.3 ms ± 273 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
40.7 ms ± 445 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
39.90ms =>  430.5 Bio. interactions per second


# Profile Force

In [ ]:
x = jax.random.normal(jax.random.PRNGKey(0), (32*1024, 3)).block_until_ready()
%timeit -n 10 force_via_matrix(x).block_until_ready()  # For the jax implementation we can't go beyond 32k -> it runs out of memory.
%timeit -n 10 ffiforce(x, block_size=64).block_until_ready()

41.6 ms ± 10.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
6.11 ms ± 223 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
x = jax.random.normal(jax.random.PRNGKey(0), (128*1024, 3)).block_until_ready()
print("Potential:")
%timeit -n 10 forces.potential_jit(x, block_size=64).block_until_ready()
print("Force:")
%timeit -n 10 forces.force_jit(x, block_size=64).block_until_ready()
%timeit -n 10 forces.force_jit(x, block_size=128).block_until_ready()

t0 = time.time()
for i in range(10):
    forces.force_jit(x, block_size=64).block_until_ready()
t1 = time.time()

dt = (t1-t0) / 10
print(f"{dt*1e3:.1f}ms =>  {(128*1024)**2 / dt  / 1e9:.1f} Bio. interactions per second", )

Potential:
37.3 ms ± 8.75 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
Force:
46.4 ms ± 250 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
